In [ ]:
import os

from pathlib import Path
from matplotlib import pyplot as plt
plt.style.use(Path(os.environ["HOME"]) / "default.mplstyle")

from astropy import units as au
from astropy import constants as ac
import xarray as xr
import numpy as np

if "FRIGUS_DATADIR_ROOT" not in os.environ:
    os.environ["FRIGUS_DATADIR_ROOT"] = f"{os.environ['CONDA_PREFIX']}/data"
from LWphotorates.utils import spec_nu2lambda, nu2lambda
from LWphotorates.stellar_spectra.utils import DATA_DIR, normalise_blackbody_spectrum, OUT_STELLAR_MASS

# Create the spectra and find the correct normalisation

In [ ]:
POPIII_BB_TEMP = 1e5 * au.K
POPII_BB_TEMP = 1e4 * au.K
POPIII_PHOTON_COUNT_PER_BARYON = 2e4
POPII_PHOTON_COUNT_PER_BARYON = 4e3

popIII_energy_array, popIII_bb_spectrum = normalise_blackbody_spectrum(
    POPIII_BB_TEMP, POPIII_PHOTON_COUNT_PER_BARYON)
popII_energy_array, popII_bb_spectrum = normalise_blackbody_spectrum(
    POPII_BB_TEMP, POPII_PHOTON_COUNT_PER_BARYON)

Quick plot just to show T5 and T4 spectra.

In [ ]:
fig, ax = plt.subplots()
ax.plot(popIII_energy_array, popIII_bb_spectrum)
ax.plot(popII_energy_array, popII_bb_spectrum)
ax.set_yscale("log")
ax.set_xlabel(r"$\epsilon$  [eV]")
ax.set_ylabel(r"L$_\nu$  [erg s$^{-1}$ Hz$^{-1}$]")

In [ ]:
frequency_array = popIII_energy_array / ac.h.to(au.eV / au.Hz)
wavelength_array = nu2lambda(frequency_array)

popIII_bb_spectrum_lambda = spec_nu2lambda(frequency_array, popIII_bb_spectrum)
popII_bb_spectrum_lambda = spec_nu2lambda(frequency_array, popII_bb_spectrum)

Compare the T5 and T4 spectra with some examples from BPASS and Yggdrasil.

In [ ]:
bpass_example_spec = xr.open_dataset(DATA_DIR / "bpass" / "spectra-bin-imf100_300.zem5.nc")
yggdrasil_example_spec = xr.open_dataset(DATA_DIR / "Yggdrasil_PopIII_Kr.nc")

fig, ax = plt.subplots()
ax.plot(wavelength_array, popIII_bb_spectrum_lambda)
ax.plot(wavelength_array, popII_bb_spectrum_lambda)
ax.plot(bpass_example_spec.wavelength, bpass_example_spec.luminosity.isel(age=0))
ax.plot(yggdrasil_example_spec.wavelength, yggdrasil_example_spec.luminosity.isel(age=90))
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_ylim(bottom=1e32, top=1e42)
ax.set_xlim(1e2, 1e5)
ax.set_xlabel(r"$\lambda$  [$\AA$]")
ax.set_ylabel(r"L$_\lambda$  [erg s$^{-1}$ $\AA^{-1}$]")

# Save the spectra, following the same structure as for Yggdrasil and BPASS

In [ ]:
ds_description = (
    "10^5 K black-body spectrum, normalised such that the LW photons emitted per stellar baryon amounts to 2e4 "
    "(Greif & Bromm, 2006).")

out_file_path = DATA_DIR / "T5_spectra.nc"

ds = xr.Dataset(
    data_vars=dict(
        luminosity=(
            ["age", "wavelength"], np.repeat(np.expand_dims(popIII_bb_spectrum_lambda, 0), 2, 0),
            {"units": (au.erg / au.s / au.AA).to_string(), "stellar_mass": OUT_STELLAR_MASS.value}),
    ),
    coords=dict(
        age=(["age"], [0.01, 5], {"units": au.Myr.to_string()}),
        wavelength=(["wavelength"], wavelength_array, {"units": au.AA.to_string()}),
    ),
    attrs=dict(description=ds_description)
)

ds.to_netcdf(out_file_path)

In [ ]:
ds_description = (
    "10^4 K black-body spectrum, normalised such that the LW photons emitted per stellar baryon amounts to 4e3 "
    "(Greif & Bromm, 2006).")

out_file_path = DATA_DIR / "T4_spectra.nc"

ds = xr.Dataset(
    data_vars=dict(
        luminosity=(
            ["age", "wavelength"], np.repeat(np.expand_dims(popII_bb_spectrum_lambda, 0), 2, 0),
            {"units": (au.erg / au.s / au.AA).to_string(), "stellar_mass": OUT_STELLAR_MASS.value}),
    ),
    coords=dict(
        age=(["age"], [0.01, 5], {"units": au.Myr.to_string()}),
        wavelength=(["wavelength"], wavelength_array, {"units": au.AA.to_string()}),
    ),
    attrs=dict(description=ds_description)
)

ds.to_netcdf(out_file_path)

Check that the spectra are correct.

In [ ]:
t5_ds = xr.open_dataset(DATA_DIR / "T5_spectra.nc")
t4_ds = xr.open_dataset(DATA_DIR / "T4_spectra.nc")

In [ ]:
bpass_example_spec = xr.open_dataset(DATA_DIR / "bpass" / "spectra-bin-imf100_300.zem5.nc")
yggdrasil_example_spec = xr.open_dataset(DATA_DIR / "Yggdrasil_PopIII_Kr.nc")

fig, ax = plt.subplots()
ax.plot(t5_ds.wavelength, t5_ds.luminosity.isel(age=0))
ax.plot(t4_ds.wavelength, t4_ds.luminosity.isel(age=0))
ax.plot(bpass_example_spec.wavelength, bpass_example_spec.luminosity.isel(age=0))
ax.plot(yggdrasil_example_spec.wavelength, yggdrasil_example_spec.luminosity.isel(age=90))
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_ylim(bottom=1e32, top=1e42)
ax.set_xlim(1e2, 1e5)
ax.set_xlabel(r"$\lambda$  [$\AA$]")
ax.set_ylabel(r"L$_\lambda$  [erg s$^{-1}$ $\AA^{-1}$]")